<a href="https://colab.research.google.com/github/h77k/python-ai-Trunova-Polina/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2b: Data Analysis — Чтение, диагностика и подготовка выборок

**Цель:**
Научиться читать CSV-файлы из репозитория GitHub, выполнять строгую диагностику качества данных (проверка на дубликаты, заполненность, типы объектов) и готовить обогащенные выборки для визуализации, включая объединение с данными о производителях.

**Данные:**
1.  `data/models.csv` — информация о моделях техники (название, тип, производитель, страна, даты, габариты, масса, связи поколений).
2.  `data/manufacturers.csv` — информация о производителях (год основания, отрасль, штаб-квартира, веб-сайт).

**Что мы делаем:**
1.  **Подготовка среды:** Клонируем репозиторий и импортируем все необходимые библиотеки в начале ноутбука.
2.  **Чтение и первичная очистка:**
    *   Загружаем оба CSV-файла.
    *   Переименовываем столбцы (сохраняем `URL`, стандартизируем имена меток).
    *   Приводим числовые поля и даты к корректным типам (`pd.to_numeric`, `pd.to_datetime`).
3.  **Строгая диагностика данных:**
    *   Проверяем соотношение "строки / уникальные модели", чтобы выявить проблемы длинного формата (когда одна модель имеет несколько типов).
    *   Рассчитываем **Fill Rate** (долю заполненности) для всех OPTIONAL-полей, чтобы объективно решить, какие графики имеют смысл.
    *   Анализируем распределение типов техники (`type_name`) для выделения основного ядра данных (`df_core`).
4.  **Обогащение данных:**
    *   Выполняем `merge` с таблицей производителей (`manufacturers.csv`) для получения дополнительных признаков (год основания компании, отрасль).
5.  **Формирование предметных выборок:**
    *   Создаем специализированные датафреймы (`df_time`, `df_dims`, `df_mass`, `df_lineage`) с расчетом производных признаков (например, `footprint`, `slenderness`), четко фиксируя размер выборки (N) для каждого будущего графика.

## 🐱 [1] Клонируем репозиторий курса в Colab

In [1]:
# 🐱 Шаг 1. Импорт библиотек и клонирование репозитория

# 1. Импорт всех необходимых библиотек (требование курса: все импорты в начале)
import os
import pandas as pd
import numpy as np

# 2. Клонирование репозитория проекта в Colab
repo = "python-ai-Trunova-Polina"
repo_path = f"/content/{repo}"  # абсолютный путь — не зависит от cwd

# Клонируем, если папки еще нет
if not os.path.exists(repo_path):
    !git clone -q https://github.com/h77k/python-ai-Trunova-Polina.git

# Переходим в папку репозитория, если мы еще не там
if os.getcwd() != repo_path:
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)
print("✅ Библиотеки pandas и numpy импортированы")

/content/python-ai-Trunova-Polina
✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-Trunova-Polina
✅ Библиотеки pandas и numpy импортированы


# 📥 [2A] Чтение CSV-файлов в pandas

На этом этапе мы считываем оба исходных файла в отдельные DataFrame без изменений, чтобы зафиксировать исходное состояние данных.

**Файлы:**
1.  `data/models.csv` — основной датасет с моделями техники.
2.  `data/manufacturers.csv` — справочник производителей.

После чтения мы проверим размер каждого датасета (количество строк) и визуально оценим первые строки, чтобы понять структуру столбцов перед началом очистки.

In [2]:
# 🐱 Шаг 2A. Чтение CSV-файлов в pandas

# 1. Читаем основной файл с моделями
df = pd.read_csv("data/models.csv")
print("✅ Загружено строк в df (models):", len(df))

# 2. Читаем файл с производителями (требование курса: работать с обоими файлами)
df_mfr = pd.read_csv("data/manufacturers.csv")
print("✅ Загружено строк в df_mfr (manufacturers):", len(df_mfr))

# Быстрый взгляд на данные
print("\n--- Первые строки models ---")
display(df.head(2))
print("\n--- Первые строки manufacturers ---")
display(df_mfr.head(2))

✅ Загружено строк в df (models): 1929
✅ Загружено строк в df_mfr (manufacturers): 18

--- Первые строки models ---


,model,modelLabel,type,typeLabel,manufacturer,manufacturerLabel,countryLabel,startProduction,predecessorLabel,successorLabel,length,width,height,mass,image
0,http://www.wikidata.org/entity/Q91031140,BMW Z3 M (E36/7),http://www.wikidata.org/entity/Q12627468,Q12627468,http://www.wikidata.org/entity/Q26678,BMW,Германия,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://commons.wikimedia.org/wiki/Special:File...
1,http://www.wikidata.org/entity/Q90996924,BMW 2000 touring,http://www.wikidata.org/entity/Q67388842,BMW 2000,http://www.wikidata.org/entity/Q26678,BMW,Германия,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://commons.wikimedia.org/wiki/Special:File...



--- Первые строки manufacturers ---


,manufacturer,manufacturerLabel,countryLabel,hqLabel,inception,industryLabel,website,logo,image
0,http://www.wikidata.org/entity/Q26678,BMW,Германия,Мюнхен,1916-03-07T00:00:00Z,автомобильная промышленность,https://www.bmw.com,http://commons.wikimedia.org/wiki/Special:File...,http://commons.wikimedia.org/wiki/Special:File...
1,http://www.wikidata.org/entity/Q26678,BMW,Германия,Мюнхен,1916-03-07T00:00:00Z,авиастроение,https://www.bmw.com,http://commons.wikimedia.org/wiki/Special:File...,http://commons.wikimedia.org/wiki/Special:File...


# 🧹 [2B] Очистка и переименование столбцов

В этом шаге мы нормализуем оба датафрейса, чтобы подготовить их к анализу и объединению.

**1. Очистка `df` (models.csv):**
*   **Переименование:**
    *   `model` → `URL` (сохраняем ссылку, как требует задание).
    *   `*Label` → понятные имена (`model_name`, `type_name`, `manufacturer_name`, `country_name`).
    *   `startProduction` → `start_production`.
*   **Числа:** Поля `length`, `width`, `height`, `mass` приводим к числу через `pd.to_numeric(errors='coerce')`.
*   **Даты:** Из `start_production` извлекаем `start_year`.

**2. Очистка `df_mfr` (manufacturers.csv):**
*   **Переименование:** Приводим имена столбцов к виду, удобному для слияния с основным табличей.
    *   `manufacturer` → `manufacturer_url` (ключ для связи).
    *   `manufacturerLabel` → `manufacturer_name`.
    *   `inception` → `manufacturer_inception` (дата основания).
    *   `industryLabel` → `industry_name`.
*   **Даты:** Из `manufacturer_inception` извлекаем год основания компании (`manufacturer_inception_year`).

**Важно:** Мы не заполняем пропуски нулями (`fillna(0)`), а оставляем `NaN`, чтобы честно оценить заполненность данных на следующем шаге.

In [3]:
# 🧹 Шаг 2B. Очистка и переименование столбцов

# --- ЧАСТЬ 1: Очистка основного датасета (models) ---
if "modelLabel" in df.columns:
    # 1. Переименование столбцов
    df = df.rename(columns={
        'model': 'URL',
        'modelLabel': 'model_name',
        'typeLabel': 'type_name',
        'manufacturerLabel': 'manufacturer_name',
        'countryLabel': 'country_name',
        'startProduction': 'start_production'
    })

    # 2. Приведение числовых полей
    numeric_cols = ['length', 'width', 'height', 'mass']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # 3. Обработка даты производства
    df['start_production'] = pd.to_datetime(df['start_production'], errors='coerce')
    df['start_year'] = df['start_production'].dt.year

    print("✅ DataFrame df (models) очищен")
else:
    print("⏭️ DataFrame df уже очищен")

# --- ЧАСТЬ 2: Очистка датасета производителей (manufacturers) ---
if "manufacturerLabel" in df_mfr.columns:
    # 1. Переименование для будущего merge
    df_mfr = df_mfr.rename(columns={
        'manufacturer': 'manufacturer_url',       # Ключ связи
        'manufacturerLabel': 'manufacturer_name', # Имя для отображения
        'countryLabel': 'mfr_country',            # Страна штаб-квартиры
        'hqLabel': 'hq_name',                     # Штаб-квартира
        'inception': 'manufacturer_inception',    # Дата основания
        'industryLabel': 'industry_name'          # Отрасль
    })

    # 2. Извлечение года основания компании
    df_mfr['manufacturer_inception'] = pd.to_datetime(df_mfr['manufacturer_inception'], errors='coerce')
    df_mfr['manufacturer_inception_year'] = df_mfr['manufacturer_inception'].dt.year

    print("✅ DataFrame df_mfr (manufacturers) очищен")
else:
    print("⏭️ DataFrame df_mfr уже очищен")

print("\n✅ Оба датафрейма готовы к диагностике и слиянию")

✅ DataFrame df (models) очищен
✅ DataFrame df_mfr (manufacturers) очищен

✅ Оба датафрейма готовы к диагностике и слиянию


# 🔍 [3] Обзор данных: структура, уникальность и типы

После первичной очистки проведем углубленную диагностику обоих датафреймов.

**Для основного датасета `df` (models):**
1.  **Структура:** Размер таблицы (`shape`), список столбцов и типы данных (`dtypes`). Важно убедиться, что числовые поля (`length`, `mass`) имеют тип `float64` (из-за возможных `NaN`), а год — числовой.
2.  **Диагностика "Длинного формата" (ВАЖНО):**
    *   Сравним общее количество строк с количеством уникальных моделей (`URL`).
    *   Если строк значительно больше, чем уникальных URL, значит, одна модель представлена несколькими строками (например, имеет несколько типов `type_name`). Это критично для корректного подсчета статистик.
3.  **Статистика:** `describe()` для числовых колонок, чтобы найти аномалии (отрицательная длина, нереальная масса).
4.  **Первые строки:** Визуальная проверка корректности переименования.

**Для датасета `df_mfr` (manufacturers):**
1.  Краткий обзор структуры и первых строк, чтобы убедиться в корректности извлечения года основания компании.

Используем функцию `show_info` для стандартизированного вывода.

In [4]:
def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, типы данных и первые строки."""
    print(f"\n📊 {name}")
    print("-" * 30)
    print("Размер (строки, столбцы):", df.shape)
    print("\nТипы данных (dtypes):")
    print(df.dtypes)
    print("\nПервые строки:")
    display(df.head(n))  # display лучше форматирует таблицы в Colab

# 🔍 Шаг 3. Обзор данных

# 1. Обзор основного датасета
show_info(df, "Модели техники (df)")

# 2. Диагностика уникальности моделей (Проверка на "длинный формат")
print("\n🔎 Диагностика уникальности моделей:")
total_rows = len(df)
unique_models = df['URL'].nunique()
print(f"Всего строк: {total_rows}")
print(f"Уникальных моделей (по URL): {unique_models}")
print(f"Среднее число строк на модель: {round(total_rows / unique_models, 2)}")

if total_rows > unique_models:
    print("⚠️ Внимание: Одна модель может иметь несколько строк (разные типы).")
    print("   Для статистик по 'количеству моделей' нужно использовать drop_duplicates(subset='URL').")
else:
    print("✅ Структура плоская: одна строка = одна модель.")

# 3. Обзор датасета производителей
show_info(df_mfr, "Производители (df_mfr)")


📊 Модели техники (df)
------------------------------
Размер (строки, столбцы): (1929, 16)

Типы данных (dtypes):
URL                               object
model_name                        object
type                              object
type_name                         object
manufacturer                      object
manufacturer_name                 object
country_name                      object
start_production     datetime64[ns, UTC]
predecessorLabel                  object
successorLabel                    object
length                           float64
width                            float64
height                           float64
mass                             float64
image                             object
start_year                       float64
dtype: object

Первые строки:


,URL,model_name,type,type_name,manufacturer,manufacturer_name,country_name,start_production,predecessorLabel,successorLabel,length,width,height,mass,image,start_year
0,http://www.wikidata.org/entity/Q91031140,BMW Z3 M (E36/7),http://www.wikidata.org/entity/Q12627468,Q12627468,http://www.wikidata.org/entity/Q26678,BMW,Германия,NaT,NaN,NaN,NaN,NaN,NaN,NaN,http://commons.wikimedia.org/wiki/Special:File...,NaN
1,http://www.wikidata.org/entity/Q90996924,BMW 2000 touring,http://www.wikidata.org/entity/Q67388842,BMW 2000,http://www.wikidata.org/entity/Q26678,BMW,Германия,NaT,NaN,NaN,NaN,NaN,NaN,NaN,http://commons.wikimedia.org/wiki/Special:File...,NaN
2,http://www.wikidata.org/entity/Q108544556,EMW 325,http://www.wikidata.org/entity/Q137188246,military vehicle model,http://www.wikidata.org/entity/Q26678,BMW,Германия,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,http://www.wikidata.org/entity/Q796496,BMW F 800,http://www.wikidata.org/entity/Q71310524,motorcycle model series,http://www.wikidata.org/entity/Q26678,BMW,Германия,NaT,NaN,NaN,NaN,NaN,NaN,NaN,http://commons.wikimedia.org/wiki/Special:File...,NaN
4,http://www.wikidata.org/entity/Q796507,BMW GS,http://www.wikidata.org/entity/Q71310524,motorcycle model series,http://www.wikidata.org/entity/Q26678,BMW,Германия,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



🔎 Диагностика уникальности моделей:
Всего строк: 1929
Уникальных моделей (по URL): 1666
Среднее число строк на модель: 1.16
⚠️ Внимание: Одна модель может иметь несколько строк (разные типы).
   Для статистик по 'количеству моделей' нужно использовать drop_duplicates(subset='URL').

📊 Производители (df_mfr)
------------------------------
Размер (строки, столбцы): (18, 10)

Типы данных (dtypes):
manufacturer_url                            object
manufacturer_name                           object
mfr_country                                 object
hq_name                                     object
manufacturer_inception         datetime64[ns, UTC]
industry_name                               object
website                                     object
logo                                        object
image                                       object
manufacturer_inception_year                  int32
dtype: object

Первые строки:


,manufacturer_url,manufacturer_name,mfr_country,hq_name,manufacturer_inception,industry_name,website,logo,image,manufacturer_inception_year
0,http://www.wikidata.org/entity/Q26678,BMW,Германия,Мюнхен,1916-03-07 00:00:00+00:00,автомобильная промышленность,https://www.bmw.com,http://commons.wikimedia.org/wiki/Special:File...,http://commons.wikimedia.org/wiki/Special:File...,1916
1,http://www.wikidata.org/entity/Q26678,BMW,Германия,Мюнхен,1916-03-07 00:00:00+00:00,авиастроение,https://www.bmw.com,http://commons.wikimedia.org/wiki/Special:File...,http://commons.wikimedia.org/wiki/Special:File...,1916
2,http://www.wikidata.org/entity/Q26678,BMW,Германия,Мюнхен,1916-03-07 00:00:00+00:00,manufacture of motor vehicles,https://www.bmw.com,http://commons.wikimedia.org/wiki/Special:File...,http://commons.wikimedia.org/wiki/Special:File...,1916
3,http://www.wikidata.org/entity/Q26678,BMW,Германия,Мюнхен,1916-03-07 00:00:00+00:00,activities of holding companies,https://www.bmw.com,http://commons.wikimedia.org/wiki/Special:File...,http://commons.wikimedia.org/wiki/Special:File...,1916
4,http://www.wikidata.org/entity/Q246,Volkswagen,Германия,Вольфсбург,1937-05-28 00:00:00+00:00,автомобильная промышленность,https://www.vw.com/en.html,http://commons.wikimedia.org/wiki/Special:File...,http://commons.wikimedia.org/wiki/Special:File...,1937


# ✅ [4] Быстрая проверка, диагностика пригодности и обогащение данных

На этом этапе мы не просто смотрим на статистику, а проводим строгую диагностику для отбора рабочих выборок.

**1. Категориальный анализ и типы объектов:**
*   Топ стран, производителей и типов техники.
*   **Важно:** Анализ распределения `type_name`. Мы должны увидеть, есть ли в данных "шум" (единичные экземпляры, не относящиеся к серийной технике), и выделить основное ядро (`df_core`).

**2. Диагностика пригодности полей (Fill Rate & Flags):**
*   Расчет доли заполненности (Fill Rate) для OPTIONAL-полей.
*   Создание бинарных флагов (`has_time`, `has_dims`, `has_mass`, `has_lineage`) для каждой строки. Это позволит нам точно знать, какая часть данных пригодна для конкретного типа графика.

**3. Обогащение данных (Merge с производителями):**
*   Объединение `df` с `df_mfr` по производителю.
*   Расчет новых признаков: например, `years_after_foundation` (сколько лет прошло от основания компании до выпуска модели). Это даст более глубокий контекст для анализа.

**4. Формирование финальных предметных выборок:**
*   Создание четких подвыборок (`df_time`, `df_dims`, `df_lineage` и др.) с расчетом производных метрик (площадь основания, вытянутость).
*   Явная фиксация размера выборки (N) для каждого датафрейма.

In [5]:
# ✅ Шаг 4. Диагностика, обогащение и формирование выборок

print("🔍 Глубокая диагностика и подготовка данных")

# --- 1. Анализ типов техники и выделение ядра (df_core) ---
print("\n🚜 Распределение типов техники (Топ-20):")
type_counts = df['type_name'].value_counts(dropna=False).head(20)
print(type_counts)

# Выделяем основные транспортные типы для чистого анализа (примерный список, можно корректировать)
# Важно: проверяем наличие этих типов в данных
main_types = ['car model', 'bus model', 'truck model', 'автомобиль', 'автобус', 'грузовой автомобиль']
# Фильтруем те типы, которые реально есть в нашем датасете
existing_main_types = [t for t in main_types if t in df['type_name'].values]

if existing_main_types:
    df_core = df[df['type_name'].isin(existing_main_types)].copy()
    print(f"\n✅ Выделено основное ядро (df_core): {len(df_core)} строк из {len(df)}")
else:
    # Если точных совпадений нет, берем все не-пустые типы как ядро для примера
    df_core = df.dropna(subset=['type_name']).copy()
    print(f"\n⚠️ Точные совпадения типов не найдены, используем все записи с типом (df_core): {len(df_core)} строк")

# --- 2. Диагностика пригодности полей (Флаги) ---
print("\n💧 Оценка пригодности данных (Флаги):")
df['has_time'] = df['start_year'].notna()
df['has_dims'] = df[['length', 'width', 'height']].notna().all(axis=1)
df['has_mass'] = df['mass'].notna()
df['has_lineage'] = df[['predecessorLabel', 'successorLabel']].notna().any(axis=1)

flags_stats = df[['has_time', 'has_dims', 'has_mass', 'has_lineage']].mean().mul(100).round(1)
print(flags_stats)
print("❗ Используйте эти проценты для понимания репрезентативности будущих графиков.")

# --- 3. Обогащение данных: Merge с производителями ---
print("\n🔗 Объединение с данными о производителях...")
# Готовим df_mfr к слиянию: берем только нужные колонки
mfr_cols = ['manufacturer_url', 'manufacturer_name', 'manufacturer_inception_year', 'industry_name']
# Переименуем manufacturer_url в manufacturer, чтобы совпало с ключом в df (если там URL производителя)
# В вашем df ключ - это столбец 'manufacturer' (URL производителя)
df_mfr_clean = df_mfr[mfr_cols].rename(columns={'manufacturer_url': 'manufacturer'})

# Делаем merge
df_merged = df.merge(df_mfr_clean, on='manufacturer', how='left')

# Расчет нового признака: возраст бренда на момент выпуска модели
df_merged['years_after_foundation'] = df_merged['start_year'] - df_merged['manufacturer_inception_year']

print(f"✅ Объединенный датасет: {len(df_merged)} строк")
print("   Новые колонки: industry_name, manufacturer_inception_year, years_after_foundation")

# --- 4. Формирование финальных предметных выборок ---
print("\n📦 Формирование рабочих выборок для Lab 3:")

# 4.1. Полная выборка (обогащенная)
df_full = df_merged.copy()

# 4.2. Временная выборка
df_time = df_full.dropna(subset=['start_year']).copy()
df_time['decade'] = (df_time['start_year'] // 10) * 10

# 4.3. Выборка габаритов (с производными признаками)
df_dims = df_full.dropna(subset=['length', 'width', 'height']).copy()
df_dims['footprint'] = df_dims['length'] * df_dims['width']      # Площадь основания
df_dims['slenderness'] = df_dims['length'] / df_dims['height']   # Вытянутость

# 4.4. Выборка массы
df_mass = df_full.dropna(subset=['mass']).copy()

# 4.5. Выборка поколений (граф связей)
df_lineage = df_full[df_full['has_lineage']].copy()

# 4.6. Основная выборка (только серийные типы)
# Используем df_core, но также обогащаем её, если нужно (здесь упрощенно берем из df_full по индексу)
# Для простоты оставим df_core как есть, но имеем в виду, что она часть df_full

# --- ИТОГОВЫЙ ОТЧЕТ ПО РАЗМЕРАМ (N) ---
print("\n📊 Размеры рабочих выборок (N):")
print(f"N full    = {len(df_full)}")
print(f"N time    = {len(df_time)}")
print(f"N dims    = {len(df_dims)}")
print(f"N mass    = {len(df_mass)}")
print(f"N lineage = {len(df_lineage)}")
print(f"N core    = {len(df_core)}")

print("\n✅ Данные полностью подготовлены к визуализации!")

🔍 Глубокая диагностика и подготовка данных

🚜 Распределение типов техники (Топ-20):
type_name
модель автомобиля               1080
модель мотоцикла                 117
концепт-кар                      116
модель грузовика                  99
модельный ряд автомобилей         79
модель двигателя                  72
модель гоночного автомобиля       36
модель автобуса                   21
модель боевой машины              16
автомобиль                        15
модель трактора                   14
рядный двигатель                  12
платформа                         12
military vehicle model            12
семейство двигателей              11
модель летательного аппарата       9
серия моделей                      9
мотоцикл                           8
Ford Fiesta                        7
семейство боевых машин             7
Name: count, dtype: int64

✅ Выделено основное ядро (df_core): 20 строк из 1929

💧 Оценка пригодности данных (Флаги):
has_time        1.6
has_dims       18.5
has_mass

# 📝 Summary

Что мы сделали в этом ноутбуке (Week 2b):

✅ **Подготовка среды:** Клонировали репозиторий и централизовали импорты.
✅ **Чтение данных:** Загрузили и предварительно очистили оба файла: `models.csv` и `manufacturers.csv`.
✅ **Строгая диагностика:**
   *   Проверили соотношение строк и уникальных моделей (выявление дубликатов по типам).
   *   Рассчитали **Fill Rate** и создали бинарные флаги пригодности данных (`has_time`, `has_dims` и др.).
   *   Проанализировали распределение типов техники и выделили основное ядро (`df_core`).
✅ **Обогащение данных:**
   *   Выполнили `merge` с таблицей производителей.
   *   Создали новые признаки: `years_after_foundation` (возраст бренда), `decade` (десятилетие), `footprint` и `slenderness` (геометрические характеристики).
✅ **Формирование предметных выборок:**
   *   Подготовили четкие датафреймы (`df_time`, `df_dims`, `df_mass`, `df_lineage`) с известным размером выборки (N).
   *   Теперь каждый будущий график в Lab 3 будет опираться на строго определенную, очищенную подвыборку.

**Готовность к Lab 3:**
Данные структурированы, проверены на шум и обогащены контекстом. Мы можем переходить к построению визуализаций, уверенно интерпретируя результаты. 🎨📈